# M3 · Evaluacion con RAGAS y Cruce con el Harness de M2

**Topicos Especiales en IA · Universidad EAFIT**  
**Responsable:** Luis  

Este notebook ejecuta la evaluacion multidimensional del sistema RAG cumpliendo con el **Nivel 4 (5 puntos)** de la rubrica:
1. **Generacion de respuestas**: Ejecuta el pipeline de generacion RAG si no existen resultados previos.
2. **Faithfulness**: Fidelidad de la respuesta generada al contexto clinico.
3. **Context Precision**: Precision de los fragmentos recuperados por el retrieval.
4. **Context Recall**: Cobertura de la evidencia necesaria para la recomendacion clinica.
5. **Answer Relevancy**: Pertinencia de la respuesta frente a la pregunta original.
6. **Cruce con el Harness de M2**: F1 de extraccion de Clinical BERT por documento vs RAG y diagnostico automatico de fallas.


## 1 · Montar Google Drive (Persistencia y Datos de M2)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/TopicosIA/Proyecto-Salud')
print('PROJECT_ROOT configurado en:', PROJECT_ROOT)

## 2 · Clonar el Repositorio (Rama `luis`)
Descarga el codigo fuente actualizado directamente desde GitHub.

In [ ]:
!rm -rf /content/Topicos-IA
!git clone --branch luis --single-branch https://github.com/luisNP21/Topicos-IA.git /content/Topicos-IA

REPO_DIR = Path('/content/Topicos-IA')
%cd {REPO_DIR}/M3/ragas

# Verificamos si existe el archivo de M2 para el cruce (en Drive o en el repo clonado)
dim1_drive = PROJECT_ROOT / 'M2/ejecucion/outputs_gold/resultado_dimension1.json'
dim1_repo  = REPO_DIR / 'M2/ejecucion/outputs_gold/resultado_dimension1.json'

if dim1_drive.exists():
    print('Resultado M2 encontrado en Google Drive.')
elif dim1_repo.exists():
    print('Resultado M2 encontrado en el repositorio clonado.')
else:
    print('Aviso: resultado_dimension1.json no encontrado (se usara respaldo integrado).')

## 3 · Instalacion de Dependencias
Instala los paquetes necesarios tanto para el modulo de generacion como para la evaluacion con RAGAS.

In [ ]:
%pip install -q -r requirements.txt -r ../generacion/requirements.txt
print('Dependencias de Generacion y RAGAS instaladas correctamente.')

## 4 · Configuracion de Secretos y Entorno (.env)
Carga la clave `GROQ_API_KEY` desde los Secretos de Colab (panel lateral de claves) tanto para la generacion como para RAGAS.

In [ ]:
import os
from google.colab import userdata

try:
    groq_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = groq_key
    print('GROQ_API_KEY configurada con exito en el entorno.')
except Exception:
    print('Aviso: GROQ_API_KEY no encontrada en Secretos. Para llamadas al LLM agregala en el panel de secretos de Colab.')

# Escribir archivo .env para que los modulos CLI tomen las variables
with open('/content/Topicos-IA/.env', 'w') as f:
    f.write(f'PROJECT_ROOT={PROJECT_ROOT}\n')
    if 'GROQ_API_KEY' in os.environ and os.environ['GROQ_API_KEY']:
        f.write(f'GROQ_API_KEY={os.environ["GROQ_API_KEY"]}\n')

## 5 · Generacion de Resultados RAG (Modulo M3/generacion)
Verifica si ya existe el archivo `resultado_generacion.json`. Si no existe, ejecuta el pipeline de generacion sobre las entidades de prueba clinicas utilizando el LLM via Groq y los contextos recuperados.

In [ ]:
import json
from pathlib import Path

# Comprobar si ya existen resultados generados previamente en Drive o en el repo
gen_out_drive = PROJECT_ROOT / 'M3/outputs/resultado_generacion.json'
gen_out_repo  = REPO_DIR / 'M3/outputs/resultado_generacion.json'

if not gen_out_drive.exists() and not gen_out_repo.exists():
    print('resultado_generacion.json no encontrado. Ejecutando modulo de generacion...')
    (REPO_DIR / 'M3/outputs').mkdir(parents=True, exist_ok=True)
    !python /content/Topicos-IA/M3/generacion/run_generacion.py --config /content/Topicos-IA/M3/generacion/config.yaml

# Identificar la ruta activa del archivo generado
target_file = gen_out_drive if gen_out_drive.exists() else gen_out_repo

if target_file.exists():
    with open(target_file, 'r', encoding='utf-8') as f:
        gen_data = json.load(f)
    casos = gen_data.get('resultados', [])
    print(f'Archivo de generacion cargado desde: {target_file}')
    print(f'Total de casos clinicos generados: {len(casos)}')
    for i, c in enumerate(casos, 1):
        ent = c.get('entidad', '')
        n_ctx = len(c.get('contexts', []))
        ans = c.get('answer', '')
        print(f'  [{i}] Entidad: {ent} | Contextos: {n_ctx} | Longitud respuesta: {len(ans)} chars')
else:
    print('Aviso: No se encontro resultado_generacion.json. La evaluacion usara el set simulado integrado.')

## 6 · Evaluacion RAGAS en Modo Mock (Rapido y Local)
Calcula faithfulness, context precision, context recall y answer relevancy usando embeddings multilingües locales y genera la tabla de cruce diagnostico con el F1 de M2.

In [ ]:
!python run_ragas.py --config config.yaml --modo mock

## 7 · Evaluacion RAGAS en Modo Real (RAGAS Oficial + Juez Groq)
Ejecuta la libreria oficial `ragas` con el LLM de Groq (`openai/gpt-oss-120b`) evaluando directamente las respuestas y contextos clinicos generados, y cruza los resultados con el harness de M2.

In [ ]:
!python run_ragas.py --config config.yaml --modo real